In [1]:
import jax
import jax.nn as jnn
import jax.numpy as jnp
import numpy as np

import diffrax
import equinox as eqx
import optax
import matplotlib.pyplot as plt
from exciting_environments.pmsm.pmsm_env import PMSM,step_eps
from functools import partial
jax.config.update("jax_enable_x64", True) 

2025-03-26 13:43:32.140325: W external/xla/xla/service/gpu/nvptx_compiler.cc:760] The NVIDIA driver's CUDA version is 12.2 which is older than the ptxas CUDA version (12.6.77). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [2]:
gpus = jax.devices()
jax.config.update("jax_default_device", gpus[0])

In [3]:
BATCH_SIZE=150
motor_env_sat = PMSM(
    saturated=True,
    LUT_motor_name="BRUSA",
    batch_size=BATCH_SIZE,
    )
motor_env_not_sat = PMSM(
    saturated=False,
    LUT_motor_name="BRUSA",
    batch_size=BATCH_SIZE,
    )

In [4]:
@eqx.filter_value_and_grad
def test_grad(actions,env):
    _,init_state = env.reset(env.env_properties)

    def body_fun(carry, action):
        state = carry
        obs, state = env.step(state, action, env.env_properties)
        return state, obs[0:2]

    _, i_dqs = jax.lax.scan(body_fun, init_state, actions)

    return jnp.sum(jnp.linalg.norm(jnp.array(i_dqs),axis=1))


In [5]:
@eqx.filter_value_and_grad
def test_grad_vmap(actions,env):
    init_state = env.vmap_init_state()

    def body_fun(carry, action):
        state = carry
        obs, state = env.vmap_step(state, action)
        return state, obs[:,0:2]

    _, i_dqs = jax.lax.scan(body_fun, init_state, actions)

    return jnp.sum(jnp.linalg.norm(jnp.array(i_dqs),axis=2))

In [6]:
sequence_len=100
actions=jnp.repeat(jnp.array([0.03,0.03])[:,None],sequence_len,axis=1).T
#val,grad=test_grad(actions,motor_env_sat)

In [7]:
import os
os.chdir("..")
from utils.evaluation import steps_eval
from policy.policy_training import DPCTrainer
import jax_dataclasses as jdc
from models.models import MLP,NeuralEulerODE

In [8]:
class ExpertModel(eqx.Module):
    motor_env: PMSM = eqx.field(static=True)
    psi_dq_mlp: MLP
    env_properties: jdc.pytree_dataclass
    in_axes_env_properties: jdc.pytree_dataclass

    def __init__(self, motor_env, psi_layer_sizes, key):
        self.motor_env = motor_env
        key, subkey = jax.random.split(key)
        self.psi_dq_mlp = MLP(
            psi_layer_sizes, key=subkey, hidden_activation=jax.nn.swish, output_activation=jax.nn.tanh
        )
        self.env_properties= motor_env.env_properties
        self.in_axes_env_properties = motor_env.in_axes_env_properties

    def __call__(self, init_obs, actions, tau):

        def body_fun(carry, action):
            obs = carry
            obs = self.step(obs, action, tau)
            return obs, obs

        _, observations = jax.lax.scan(body_fun, init_obs, actions)
        observations = jnp.concatenate([init_obs[None, :], observations], axis=0)
        return observations

    # def step(self, obs, action, tau):
    #     obs1, _ = self.motor_env.reset(self.motor_env.env_properties)  #
    #     obs1 = obs1.at[2].set((3 * 1500 / 60 * 2 * jnp.pi) / (2 * jnp.pi * 3 * 11000 / 60))
    #     obs1 = obs1.at[0].set(obs[0])
    #     obs1 = obs1.at[1].set(obs[1])
    #     obs1 = obs1.at[4].set(obs[2])
    #     obs1 = obs1.at[5].set(obs[3])
    #     state = self.motor_env.generate_state_from_observation(obs1, self.motor_env.env_properties)
    #     # obs,_= self.motor_env.step(state, action, self.motor_env.env_properties)
    #     obs, _ = self.step_expert(state, action, self.motor_env.env_properties)
    #     return jnp.concatenate([obs[0:2], obs[4:6]])

    @partial(jax.jit, static_argnums=[0, 3])
    def ode_step(self, state, u_dq, properties):
        """Computes state by simulating one step.

        Args:
            system_state: The state from which to calculate state for the next step.
            u_dq: The action to apply to the environment.
            properties: Parameters and settings of the environment, that do not change over time.

        Returns:
            state: The computed state after the one step simulation.
        """
        system_state = state.physical_state
        omega_el = system_state.omega_el
        i_d = system_state.i_d
        i_q = system_state.i_q
        eps = system_state.epsilon

        args = (u_dq, properties.static_params)
        if properties.saturated:

            def vector_field(t, y, args):
                i_d, i_q = y
                u_dq, _ = args

                J_k = jnp.array([[0, -1], [1, 0]])
                i_dq = jnp.array([i_d, i_q])
                # p_d = {q: interp(jnp.array([i_d, i_q])) for q, interp in self.motor_env.LUT_interpolators.items()}
                i_dq_norm = i_dq / properties.physical_constraints.i_d
                p_d={}
                p_d["Psi_d"] = self.Psi_d(i_dq_norm)
                p_d["Psi_q"] = self.Psi_q(i_dq_norm)
                p_d["L_dd"] = self.L_dd(i_dq_norm)
                p_d["L_dq"] = self.L_dq(i_dq_norm)
                p_d["L_qd"] = self.L_qd(i_dq_norm)
                p_d["L_qq"] = self.L_qq(i_dq_norm)

                L_diff = jnp.column_stack([p_d[q] for q in ["L_dd", "L_dq", "L_qd", "L_qq"]]).reshape(2, 2)
                L_diff_inv = jnp.linalg.inv(L_diff)
                psi_dq = jnp.column_stack([p_d[psi] for psi in ["Psi_d", "Psi_q"]]).reshape(-1)
                di_dq_1 = jnp.einsum(
                    "ij,j->i",
                    (-L_diff_inv * properties.static_params.r_s),
                    i_dq,
                )
                di_dq_2 = jnp.einsum("ik,k->i", L_diff_inv, u_dq)
                di_dq_3 = jnp.einsum("ij,jk,k->i", -L_diff_inv, J_k, psi_dq) * omega_el
                i_dq_diff = di_dq_1 + di_dq_2 + di_dq_3
                d_y = i_dq_diff[0], i_dq_diff[1]

                return d_y

        else:

            def vector_field(t, y, args):
                i_d, i_q = y
                u_dq, params = args
                u_d = u_dq[0]
                u_q = u_dq[1]
                l_d = params.l_d
                l_q = params.l_q
                psi_p = params.psi_p
                r_s = params.r_s
                i_d_diff = (u_d + omega_el * l_q * i_q - r_s * i_d) / l_d
                i_q_diff = (u_q - omega_el * (l_d * i_d + psi_p) - r_s * i_q) / l_q
                d_y = i_d_diff, i_q_diff
                return d_y

        term = diffrax.ODETerm(vector_field)
        t0 = 0
        t1 = self.motor_env.tau
        y0 = tuple([i_d, i_q])
        env_state = self.motor_env._solver.init(term, t0, t1, y0, args)
        y, _, _, env_state, _ = self.motor_env._solver.step(term, t0, t1, y0, args, env_state, made_jump=False)

        i_d_k1 = y[0]
        i_q_k1 = y[1]

        if properties.saturated:
            torque = jnp.array(
                [self.motor_env.currents_to_torque_saturated(i_d=i_d_k1, i_q=i_q_k1, env_properties=properties)]
            )[0]
        else:
            torque = jnp.array([self.motor_env.currents_to_torque(i_d_k1, i_q_k1, properties)])[0]

        with jdc.copy_and_mutate(system_state, validate=False) as system_state_next:
            system_state_next.epsilon = step_eps(eps, omega_el, self.motor_env.tau, 1.0)
            system_state_next.i_d = i_d_k1
            system_state_next.i_q = i_q_k1
            system_state_next.torque = torque  # [0]

        with jdc.copy_and_mutate(state, validate=False) as state_next:
            state_next.physical_state = system_state_next
        return state_next

    def reset(self, env_properties, vmap_helper=None):
        return self.motor_env.reset(env_properties, vmap_helper=vmap_helper)

    @partial(jax.jit, static_argnums=[0, 3])
    def step(self, state, action, env_properties):
        """Computes state by simulating one step taking the deadtime into account.

        Args:
            system_state: The state from which to calculate state for the next step.
            action: The action to apply to the environment.
            properties: Parameters and settings of the environment, that do not change over time.

        Returns:
            state: The computed state after the one step simulation.
        """

        action = self.motor_env.constraint_denormalization(action, state, env_properties)

        action_buffer = jnp.array([state.physical_state.u_d_buffer, state.physical_state.u_q_buffer])

        if env_properties.static_params.deadtime > 0:

            updated_buffer = jnp.array([action[0], action[1]])
            u_dq = action_buffer
        else:
            updated_buffer = action_buffer

            u_dq = action

        next_state = self.ode_step(state, u_dq, env_properties)
        with jdc.copy_and_mutate(next_state, validate=True) as next_state_update:
            next_state_update.physical_state.u_d_buffer = updated_buffer[0]
            next_state_update.physical_state.u_q_buffer = updated_buffer[1]

        observation = self.motor_env.generate_observation(next_state_update, env_properties)
        return observation, next_state_update

    @partial(jax.jit, static_argnums=0)
    def vmap_step(self, state, action):
        # vmap single operations
        obs, state = jax.vmap(self.step, in_axes=(0, 0, self.in_axes_env_properties))(
            state, action, self.env_properties
        )
        return obs, state

    def vmap_init_state(self):
        return self.motor_env.vmap_init_state()

    def Psi_d(self, i_dq_norm):
        return self.psi_dq_mlp(i_dq_norm)[0]  #  self.motor_env.LUT_interpolators["Psi_d"](i_dq)[0]

    def Psi_q(self, i_dq_norm):
        return self.psi_dq_mlp(i_dq_norm)[1]  #   self.motor_env.LUT_interpolators["Psi_q"](i_dq)[0]

    def Psi_d_physical(self, i_dq):
        i_dq_norm = i_dq / self.motor_env.env_properties.physical_constraints.i_d
        return self.Psi_d(i_dq_norm)

    def Psi_q_physical(self, i_dq):
        i_dq_norm = i_dq / self.motor_env.env_properties.physical_constraints.i_d
        return self.Psi_q(i_dq_norm)

    def l_d_dq(self, i_dq):
        return jax.grad(self.Psi_d_physical)(i_dq)

    def l_q_dq(self, i_dq):
        return jax.grad(self.Psi_q_physical)(i_dq)

    def L_dd(self, i_dq_norm):
        i_dq = i_dq_norm * self.motor_env.env_properties.physical_constraints.i_d
        return self.l_d_dq(i_dq)[0]

    def L_dq(self, i_dq_norm):
        i_dq = i_dq_norm * self.motor_env.env_properties.physical_constraints.i_d
        return self.l_d_dq(i_dq)[1]

    def L_qd(self, i_dq_norm):
        i_dq = i_dq_norm * self.motor_env.env_properties.physical_constraints.i_d
        return self.l_q_dq(i_dq)[0]

    def L_qq(self, i_dq_norm):
        i_dq = i_dq_norm * self.motor_env.env_properties.physical_constraints.i_d
        return self.l_q_dq(i_dq)[1]

    def L_matrix(self, i_dq):
        L_dd = self.L_dd(i_dq)
        L_dq = self.L_dq(i_dq)
        L_qd = self.L_qd(i_dq)
        L_qq = self.L_qq(i_dq)
        mat = jnp.array([[L_dd, L_dq], [L_qd, L_qq]])
        return mat
    


class ExpertModelSeparate(eqx.Module):
    motor_env: PMSM = eqx.field(static=True)
    psi_dq_mlp: MLP
    L_d_mlp: MLP
    L_q_mlp: MLP
    env_properties: jdc.pytree_dataclass

    def __init__(self, motor_env, layer_sizes, key):
        self.motor_env = motor_env
        key, subkey = jax.random.split(key)
        self.psi_dq_mlp = MLP(
            layer_sizes, key=subkey, hidden_activation=jax.nn.swish, output_activation=jax.nn.tanh
        )
        key, subkey = jax.random.split(key)
        self.L_d_mlp = MLP(
            layer_sizes, key=subkey, hidden_activation=jax.nn.swish, output_activation=jax.nn.tanh
        )
        key, subkey = jax.random.split(key)
        self.L_q_mlp = MLP(
            layer_sizes, key=subkey, hidden_activation=jax.nn.swish, output_activation=jax.nn.tanh
        )
        self.env_properties= motor_env.env_properties

    def __call__(self, init_obs, actions, tau):

        def body_fun(carry, action):
            obs = carry
            obs = self.step(obs, action, tau)
            return obs, obs

        _, observations = jax.lax.scan(body_fun, init_obs, actions)
        observations = jnp.concatenate([init_obs[None, :], observations], axis=0)
        return observations

    # def step(self, obs, action, tau):
    #     obs1, _ = self.motor_env.reset(self.motor_env.env_properties)  #
    #     obs1 = obs1.at[2].set((3 * 1500 / 60 * 2 * jnp.pi) / (2 * jnp.pi * 3 * 11000 / 60))
    #     obs1 = obs1.at[0].set(obs[0])
    #     obs1 = obs1.at[1].set(obs[1])
    #     obs1 = obs1.at[4].set(obs[2])
    #     obs1 = obs1.at[5].set(obs[3])
    #     state = self.motor_env.generate_state_from_observation(obs1, self.motor_env.env_properties)
    #     # obs,_= self.motor_env.step(state, action, self.motor_env.env_properties)
    #     obs, _ = self.step_expert(state, action, self.motor_env.env_properties)
    #     return jnp.concatenate([obs[0:2], obs[4:6]])

    @partial(jax.jit, static_argnums=[0, 3])
    def ode_step(self, state, u_dq, properties):
        """Computes state by simulating one step.

        Args:
            system_state: The state from which to calculate state for the next step.
            u_dq: The action to apply to the environment.
            properties: Parameters and settings of the environment, that do not change over time.

        Returns:
            state: The computed state after the one step simulation.
        """
        system_state = state.physical_state
        omega_el = system_state.omega_el
        i_d = system_state.i_d
        i_q = system_state.i_q
        eps = system_state.epsilon

        args = (u_dq, properties.static_params)
        if properties.saturated:

            def vector_field(t, y, args):
                i_d, i_q = y
                u_dq, _ = args

                J_k = jnp.array([[0, -1], [1, 0]])
                i_dq = jnp.array([i_d, i_q])
                p_d = {q: interp(jnp.array([i_d, i_q])) for q, interp in self.motor_env.LUT_interpolators.items()}
                #i_dq_norm = i_dq / properties.physical_constraints.i_d
                # p_d={}
                # p_d["Psi_d"] = self.Psi_d(i_dq_norm)
                # p_d["Psi_q"] = self.Psi_q(i_dq_norm)
                # p_d["L_dd"] = self.L_dd(i_dq_norm)
                # p_d["L_dq"] = self.L_dq(i_dq_norm)
                # p_d["L_qd"] = self.L_qd(i_dq_norm)
                # p_d["L_qq"] = self.L_qq(i_dq_norm)

                L_diff = jnp.column_stack([p_d[q] for q in ["L_dd", "L_dq", "L_qd", "L_qq"]]).reshape(2, 2)
                L_diff_inv = jnp.linalg.inv(L_diff)
                psi_dq = jnp.column_stack([p_d[psi] for psi in ["Psi_d", "Psi_q"]]).reshape(-1)
                di_dq_1 = jnp.einsum(
                    "ij,j->i",
                    (-L_diff_inv * properties.static_params.r_s),
                    i_dq,
                )
                di_dq_2 = jnp.einsum("ik,k->i", L_diff_inv, u_dq)
                di_dq_3 = jnp.einsum("ij,jk,k->i", -L_diff_inv, J_k, psi_dq) * omega_el
                i_dq_diff = di_dq_1 + di_dq_2 + di_dq_3
                d_y = i_dq_diff[0], i_dq_diff[1]

                return d_y

        else:

            def vector_field(t, y, args):
                i_d, i_q = y
                u_dq, params = args
                u_d = u_dq[0]
                u_q = u_dq[1]
                l_d = params.l_d
                l_q = params.l_q
                psi_p = params.psi_p
                r_s = params.r_s
                i_d_diff = (u_d + omega_el * l_q * i_q - r_s * i_d) / l_d
                i_q_diff = (u_q - omega_el * (l_d * i_d + psi_p) - r_s * i_q) / l_q
                d_y = i_d_diff, i_q_diff
                return d_y

        term = diffrax.ODETerm(vector_field)
        t0 = 0
        t1 = self.motor_env.tau
        y0 = tuple([i_d, i_q])
        env_state = self.motor_env._solver.init(term, t0, t1, y0, args)
        y, _, _, env_state, _ = self.motor_env._solver.step(term, t0, t1, y0, args, env_state, made_jump=False)

        i_d_k1 = y[0]
        i_q_k1 = y[1]

        if properties.saturated:
            torque = jnp.array(
                [self.motor_env.currents_to_torque_saturated(i_d=i_d_k1, i_q=i_q_k1, env_properties=properties)]
            )[0]
        else:
            torque = jnp.array([self.motor_env.currents_to_torque(i_d_k1, i_q_k1, properties)])[0]

        with jdc.copy_and_mutate(system_state, validate=False) as system_state_next:
            system_state_next.epsilon = step_eps(eps, omega_el, self.motor_env.tau, 1.0)
            system_state_next.i_d = i_d_k1
            system_state_next.i_q = i_q_k1
            system_state_next.torque = torque  # [0]

        with jdc.copy_and_mutate(state, validate=False) as state_next:
            state_next.physical_state = system_state_next
        return state_next

    def reset(self, env_properties):
        return self.motor_env.reset(env_properties)

    @partial(jax.jit, static_argnums=[0, 3])
    def step(self, state, action, env_properties):
        """Computes state by simulating one step taking the deadtime into account.

        Args:
            system_state: The state from which to calculate state for the next step.
            action: The action to apply to the environment.
            properties: Parameters and settings of the environment, that do not change over time.

        Returns:
            state: The computed state after the one step simulation.
        """

        action = self.motor_env.constraint_denormalization(action, state, env_properties)

        action_buffer = jnp.array([state.physical_state.u_d_buffer, state.physical_state.u_q_buffer])

        if env_properties.static_params.deadtime > 0:

            updated_buffer = jnp.array([action[0], action[1]])
            u_dq = action_buffer
        else:
            updated_buffer = action_buffer

            u_dq = action

        next_state = self.ode_step(state, u_dq, env_properties)
        with jdc.copy_and_mutate(next_state, validate=True) as next_state_update:
            next_state_update.physical_state.u_d_buffer = updated_buffer[0]
            next_state_update.physical_state.u_q_buffer = updated_buffer[1]

        observation = self.motor_env.generate_observation(next_state_update, env_properties)
        return observation, next_state_update

    def Psi_d(self, i_dq_norm):
        return self.psi_dq_mlp(i_dq_norm)[0]  #  self.motor_env.LUT_interpolators["Psi_d"](i_dq)[0]

    def Psi_q(self, i_dq_norm):
        return self.psi_dq_mlp(i_dq_norm)[1]  #   self.motor_env.LUT_interpolators["Psi_q"](i_dq)[0]

    def Psi_d_physical(self, i_dq):
        i_dq_norm = i_dq / self.motor_env.env_properties.physical_constraints.i_d
        return self.Psi_d(i_dq_norm)

    def Psi_q_physical(self, i_dq):
        i_dq_norm = i_dq / self.motor_env.env_properties.physical_constraints.i_d
        return self.Psi_q(i_dq_norm)

    def l_d_dq(self, i_dq):
        return jax.grad(self.Psi_d_physical)(i_dq)

    def l_q_dq(self, i_dq):
        return jax.grad(self.Psi_q_physical)(i_dq)

    # def L_dd(self, i_dq_norm):
    #     i_dq = i_dq_norm * self.motor_env.env_properties.physical_constraints.i_d
    #     return self.l_d_dq(i_dq)[0]

    # def L_dq(self, i_dq_norm):
    #     i_dq = i_dq_norm * self.motor_env.env_properties.physical_constraints.i_d
    #     return self.l_d_dq(i_dq)[1]

    # def L_qd(self, i_dq_norm):
    #     i_dq = i_dq_norm * self.motor_env.env_properties.physical_constraints.i_d
    #     return self.l_q_dq(i_dq)[0]

    # def L_qq(self, i_dq_norm):
    #     i_dq = i_dq_norm * self.motor_env.env_properties.physical_constraints.i_d
    #     return self.l_q_dq(i_dq)[1]

    def L_dd(self, i_dq_norm):
        return self.L_d_mlp(i_dq_norm)[0]

    def L_dq(self, i_dq_norm):
        return self.L_d_mlp(i_dq_norm)[1]

    def L_qd(self, i_dq_norm):
        return self.L_q_mlp(i_dq_norm)[0]

    def L_qq(self, i_dq_norm):
        return self.L_q_mlp(i_dq_norm)[1]

    def L_matrix(self, i_dq):
        L_dd = self.L_dd(i_dq)
        L_dq = self.L_dq(i_dq)
        L_qd = self.L_qd(i_dq)
        L_qq = self.L_qq(i_dq)
        mat = jnp.array([[L_dd, L_dq], [L_qd, L_qq]])
        return mat

In [9]:
motor_env_not_sat.vmap_init_state

<bound method PMSM.vmap_init_state of <exciting_environments.pmsm.pmsm_env.PMSM object at 0x7f0c3430a510>>

In [10]:
jax_key = jax.random.PRNGKey(2)
node_grey_4_128= ExpertModel(motor_env=motor_env_sat,psi_layer_sizes=[2,128,128,128,128,2],key=jax_key)
node_grey_3_64= ExpertModel(motor_env=motor_env_sat,psi_layer_sizes=[2,64,64,64,2],key=jax_key)
node_grey_2_32= ExpertModel(motor_env=motor_env_sat,psi_layer_sizes=[2,32,32,2],key=jax_key)

In [11]:
import time
import jax.numpy as jnp
import jax

def measure_jax_time(func, actions,env, warmup=True, runs=200):
    if warmup:
        func(actions[0],env)[1].block_until_ready()  

    times = []
    for i in range(runs):
        start_time = time.perf_counter()
        result = func(actions[i],env)
        result[1].block_until_ready()
        end_time = time.perf_counter()
        
        times.append(end_time - start_time)

    return jnp.mean(jnp.array(times)), jnp.std(jnp.array(times))

jit_test_grad = eqx.filter_jit(test_grad)

actions=jax.random.uniform(key=jax.random.PRNGKey(4),shape=(200,100,2),minval=0.025,maxval=0.035)


mean_time, std_time = measure_jax_time(jit_test_grad, actions, motor_env_sat)
print(f"Saturated Env: {mean_time:.6f}s ± {std_time:.6f}s")

mean_time, std_time = measure_jax_time(jit_test_grad, actions, motor_env_not_sat)
print(f"Linear Env: {mean_time:.6f}s ± {std_time:.6f}s")

mean_time, std_time = measure_jax_time(jit_test_grad, actions, node_grey_4_128)
print(f"Grey-Box (128,128,128,128): {mean_time:.6f}s ± {std_time:.6f}s")

mean_time, std_time = measure_jax_time(jit_test_grad, actions, node_grey_3_64)
print(f"Grey-Box (64,64,64): {mean_time:.6f}s ± {std_time:.6f}s")

mean_time, std_time = measure_jax_time(jit_test_grad, actions, node_grey_2_32)
print(f"Grey-Box (32,32): {mean_time:.6f}s ± {std_time:.6f}s")


Saturated Env: 0.034556s ± 0.001950s
Linear Env: 0.006035s ± 0.000140s
Grey-Box (128,128,128,128): 0.067445s ± 0.002097s
Grey-Box (64,64,64): 0.050557s ± 0.002325s
Grey-Box (32,32): 0.041573s ± 0.001579s


## Vmapped batch_size=150

In [12]:
import time
import jax.numpy as jnp
import jax

def measure_jax_time(func, actions,env, warmup=True, runs=200):
    if warmup:
        func(actions[0],env)[1].block_until_ready()  

    times = []
    for i in range(runs):
        start_time = time.perf_counter()
        result = func(actions[i],env)
        result[1].block_until_ready()
        end_time = time.perf_counter()
        
        times.append(end_time - start_time)

    return jnp.mean(jnp.array(times)), jnp.std(jnp.array(times))

jit_test_grad = eqx.filter_jit(test_grad_vmap)

actions=jax.random.uniform(key=jax.random.PRNGKey(4),shape=(200,100,150,2),minval=0.025,maxval=0.035)


mean_time, std_time = measure_jax_time(jit_test_grad, actions, motor_env_sat)
print(f"Saturated Env: {mean_time:.6f}s ± {std_time:.6f}s")

mean_time, std_time = measure_jax_time(jit_test_grad, actions, motor_env_not_sat)
print(f"Linear Env: {mean_time:.6f}s ± {std_time:.6f}s")

mean_time, std_time = measure_jax_time(jit_test_grad, actions, node_grey_4_128)
print(f"Grey-Box (128,128,128,128): {mean_time:.6f}s ± {std_time:.6f}s")

mean_time, std_time = measure_jax_time(jit_test_grad, actions, node_grey_3_64)
print(f"Grey-Box (64,64,64): {mean_time:.6f}s ± {std_time:.6f}s")

mean_time, std_time = measure_jax_time(jit_test_grad, actions, node_grey_2_32)
print(f"Grey-Box (32,32): {mean_time:.6f}s ± {std_time:.6f}s")

Saturated Env: 0.136561s ± 0.002218s
Linear Env: 0.007528s ± 0.000104s
Grey-Box (128,128,128,128): 0.266756s ± 0.001173s
Grey-Box (64,64,64): 0.151822s ± 0.001060s
Grey-Box (32,32): 0.095385s ± 0.000753s
